# 11 — E3C: Semantic Alignment Weight Sensitivity — **FIXED v2**

**SNA-MXH / Group 14 · Google Colab · RQ2**

- **Weights:** `0, 0.005, 0.01, 0.02, 0.05`; **seeds:** `2023, 2024, 2025`.
- **12 runs mới**; 3 runs `0.01` dùng lại từ E1, không train lại.
- **FIX:** Notebook tự clone repo đúng commit khi Colab reset, tìm nguồn dữ liệu trong Drive, kiểm tra repo/data/GPU *độc lập*, tạo liên kết `data/amazon` cho training workspace; không chỉnh sửa dữ liệu E1/E3.
- **An toàn:** Mặc định `DRY_RUN=True`. Chỉ khi chủ động bật `DRY_RUN=False` mới train. Với `SMOKE_TEST_ONLY=True`, chạy *một run*.
- **Output:** checkpoints, console logs, config snapshot, status/manifest. **Chưa** tự xuất Top-K hay tính ARP/APLT/TailRecall; cần dùng exporter và analyzer E2/E3B.

> Lưu ý về tái lập: Git commit được ghim theo log E3A. Repo gốc có thể cần các bản vá tương thích PyTorch của E1; notebook sẽ hiển thị commit/checksum và kiểm tra môi trường, nhưng không tự đoán hoặc áp dụng những bản vá không được lưu lại. Nếu repo gốc khác source dùng E1, cần xác minh trước khi dùng số liệu để viết báo cáo.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import csv, json, subprocess, sys, shutil, os, hashlib, time, importlib.util, traceback
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Data_SNA_Group14')
OUT = ROOT / 'experiments' / 'E3C'
SOURCE_REPO = Path('/content/RLMRec')
REPO_URL = 'https://github.com/HKUDS/RLMRec.git'
EXPECTED_COMMIT = '22413752246de3dee8ab0d509f7f7a8889080f95'  # E3A pinned commit

WEIGHTS = (0.0, 0.005, 0.01, 0.02, 0.05)
SEEDS = (2023, 2024, 2025)
DEFAULT_WEIGHT = 0.01
MODEL, DATASET = 'lightgcn_plus', 'amazon'
K_VALUES = (10, 20)

DRY_RUN = True           # Default SAFE: tạo kế hoạch, KHÔNG train
SMOKE_TEST_ONLY = True   # Khi DRY_RUN=False: chạy 1 run chưa hoàn thành
ALLOW_CLONE = True       # FIX: Colab runtime mới có thể không có /content/RLMRec
AUTO_LINK_DATA = True    # Liên kết data Amazon từ Drive khi repo đã có
RETRY_FAILED = False     # Không tự train lại run failed nếu chưa xem log

assert ROOT.is_dir(), f'Không thấy thư mục Project trong Drive: {ROOT}. Kiểm tra mount/tên folder.'
print('Project root:', ROOT)
print('Output:', OUT)
print('Source:', SOURCE_REPO, '| auto-clone:', ALLOW_CLONE)
print('Mode:', 'DRY RUN' if DRY_RUN else ('SMOKE TEST' if SMOKE_TEST_ONLY else 'FULL SWEEP'))


## A. Kiểm tra baseline và tạo manifest

Đọc kết quả Full E3B và tái sử dụng ba file Top-K `λ=0.01` của E1. Nếu thiếu đầu vào hoặc không khớp, dừng trước khi tạo training jobs.


In [ ]:
import pandas as pd

reference = ROOT / 'experiments/E3B/popularity_bias/E3B_model_summary.csv'
if not reference.is_file():
    raise FileNotFoundError(f'Thiếu E3B summary trên Drive: {reference}')
existing = pd.read_csv(reference)
check = existing[(existing['model']=='RLMRec-Con Full') & (existing['K']==20)]
assert len(check)==1 and int(check.iloc[0]['n_seeds'])==3, 'E3B reference mismatch'
print('BASELINE Full @20, weight=0.01:')
print(check[['Recall_mean','NDCG_mean','ARP_mean','APLT_mean','TailRecall_mean','Coverage_mean']].to_string(index=False))

ref_topks = {}
for seed in SEEDS:
    p = ROOT / f'experiments/predictions/lightgcn_plus/seed_{seed}/topk_test.npz'
    if not p.is_file():
        raise FileNotFoundError(f'Không thể reuse E1 seed {seed}; thiếu {p}')
    ref_topks[seed] = p
print('E1 Full Top-K references validated:', len(ref_topks))


In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
RUNS = OUT / 'runs' / 'full'
RUNS.mkdir(parents=True, exist_ok=True)

def weight_slug(w):
    return f'{w:.3f}'.rstrip('0').rstrip('.') if w else '0'

def run_dir(w, seed):
    return RUNS / f'weight_{weight_slug(w)}' / f'seed_{seed}'

def read_json_safe(path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return {}

def has_saved_checkpoint(dest):
    # Tránh coi file status riêng lẻ là bằng chứng đã train xong.
    return bool(list(dest.rglob('*.pth')) + list(dest.rglob('*.pt'))) if dest.exists() else False

def get_run_status(weight, seed):
    if weight == DEFAULT_WEIGHT:
        return 'reused_E1'
    dest=run_dir(weight, seed)
    verification=dest/'verification.json'
    if verification.is_file():
        m=read_json_safe(verification)
        if (m.get('status')=='passed' and float(m.get('weight',-1))==weight
            and int(m.get('seed',-1))==seed and (dest/'topk/topk_test.npz').is_file()
            and has_saved_checkpoint(dest)):
            return 'passed'
        return 'needs_review'
    status_file=dest/'train_status.json'
    if status_file.is_file():
        m=read_json_safe(status_file)
        status=m.get('status','needs_review')
        if status=='trained_pending_export' and not has_saved_checkpoint(dest):
            return 'needs_review'
        if status=='running':
            return 'interrupted_review'
        return status
    if has_saved_checkpoint(dest):
        return 'needs_review'
    return 'pending'

def build_manifest():
    rows=[]
    for w in WEIGHTS:
        for seed in SEEDS:
            dest=run_dir(w, seed)
            rows.append(dict(model=MODEL, align_mode='full', dataset=DATASET,
                weight=w, seed=seed, status=get_run_status(w, seed),
                run_directory=str(dest),
                topk_reference=str(ref_topks[seed]) if w==DEFAULT_WEIGHT else ''))
    df=pd.DataFrame(rows)
    df.to_csv(OUT/'E3C_run_manifest.csv',index=False)
    return df

protocol=dict(project='SNA-MXH',experiment='E3C_Semantic_Weight_Sensitivity',
    model=MODEL,dataset=DATASET,weights=list(WEIGHTS),seeds=list(SEEDS),
    reused_weight=DEFAULT_WEIGHT,source_commit=EXPECTED_COMMIT,
    architecture={'graph_layers':3,'embedding_dimension':32},
    checkpoint_selection='validation Recall@20 as E1/E3A',
    evaluation={'K':list(K_VALUES),'ranking_mask':'train_only','validation_masked':False,
                'popularity_metrics':'E2/E3B definitions'},
    experiment_scope='full alignment only; change global kd_weight')
(OUT/'E3C_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
manifest_df=build_manifest()
print(manifest_df[['weight','seed','status']].to_string(index=False))
print('Pending new runs:',sum(manifest_df.status=='pending'))


## B. AUTO SETUP + PREFLIGHT (đã sửa)

1. Nếu thiếu `/content/RLMRec`, clone GitHub và checkout **đúng commit E3A**; không ghi đè source sẵn có.
2. Tìm `data/raw/amazon` trên Drive, kiểm tra đủ `trn_mat.pkl`, `val_mat.pkl`, `tst_mat.pkl`, `usr_emb_np.pkl`, `itm_emb_np.pkl`.
3. Nếu cho phép, liên kết `SOURCE_REPO/data/amazon` → thư mục dữ liệu thật trên Drive (chỉ khi đường dẫn đích đang trống).
4. Kiểm tra GPU và những package Python cơ bản **độc lập** với repo.
5. Nếu bất kỳ bước nào thất bại, hiển thị lý do và dừng training, **không** ghi nhận RUN PASS giả.


In [ ]:
import yaml

def sh(args, cwd=None, check=True):
    p=subprocess.run(args,cwd=cwd,text=True,capture_output=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str,args))}\n{p.stderr[-4000:]}\n{p.stdout[-1000:]}")
    return p.stdout.strip()

# (1) SOURCE CHECK – không phụ thuộc DATA/GPU
REPO_READY=False
REPO_ERROR=''
if not SOURCE_REPO.is_dir():
    if ALLOW_CLONE:
        print('SOURCE missing => clone pinned repository:', REPO_URL)
        try:
            subprocess.run(['git','clone',REPO_URL,str(SOURCE_REPO)],check=True)
            subprocess.run(['git','checkout','--detach',EXPECTED_COMMIT],cwd=SOURCE_REPO,check=True)
        except Exception as ex:
            REPO_ERROR=f'Không clone/checkout được GitHub: {ex}'
    else:
        REPO_ERROR='Repository chưa tồn tại, ALLOW_CLONE=False.'
if SOURCE_REPO.is_dir():
    try:
        rev=sh(['git','rev-parse','HEAD'],cwd=SOURCE_REPO)
        if rev!=EXPECTED_COMMIT:
            REPO_ERROR=f'Commit KHÔNG khớp E3A: {rev} != {EXPECTED_COMMIT}. Không tự đổi repo đang có.'
        elif not (SOURCE_REPO/'encoder/train_encoder.py').is_file():
            REPO_ERROR='Missing encoder/train_encoder.py'
        elif not (SOURCE_REPO/'encoder/config/modelconf/lightgcn_plus.yml').is_file():
            REPO_ERROR='Missing model config YAML'
        else:
            REPO_READY=True
            dirty=sh(['git','status','--porcelain'],cwd=SOURCE_REPO)
            print('Source commit:',rev,'| pinned:',True)
            print('Source changes:',dirty[:500] if dirty else 'clean')
    except Exception as ex:
        REPO_ERROR=f'Git/source check failed: {ex}'
print('REPO_READY:',REPO_READY, '|',REPO_ERROR or 'OK')

# (2) DATA CHECK – đọc trực tiếp Drive, không phụ thuộc REPO_READY
required_data=['trn_mat.pkl','val_mat.pkl','tst_mat.pkl','usr_emb_np.pkl','itm_emb_np.pkl']
DATA_ROOT = ROOT/'data/raw/amazon'
missing_data=[x for x in required_data if not (DATA_ROOT/x).is_file()]
DATA_READY=(len(missing_data)==0)
print('DATA_ROOT:',DATA_ROOT)
print('DATA_READY:',DATA_READY,'| missing:', missing_data if missing_data else 'none')

# (3) Link an toàn, không ghi đè data cũ
DATA_LINK_READY=False
if REPO_READY and DATA_READY:
    target=SOURCE_REPO/'data'/'amazon'
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.is_symlink():
        if target.resolve()==DATA_ROOT.resolve():
            DATA_LINK_READY=True
        else:
            print('DATA LINK BLOCKED: symlink đang trỏ nơi khác:',target.resolve())
    elif target.exists():
        # Không xóa/di chuyển dữ liệu đã tồn tại trong repo
        if all((target/x).is_file() for x in required_data):
            DATA_LINK_READY=True
            print('Existing repo dataset kept; không thay thế:',target)
        else:
            print('DATA LINK BLOCKED: repo data/amazon tồn tại nhưng thiếu file. Kiểm tra thủ công:',target)
    elif AUTO_LINK_DATA:
        target.symlink_to(DATA_ROOT,target_is_directory=True)
        DATA_LINK_READY=True
        print('Linked:',target,'->',DATA_ROOT)
    else:
        print('DATA LINK BLOCKED: AUTO_LINK_DATA=False')
print('DATA_LINK_READY:',DATA_LINK_READY)

# (4) GPU, packages – kiểm tra độc lập, báo lỗi chi tiết
GPU_READY=False
try:
    import torch
    GPU_READY=bool(torch.cuda.is_available())
    print('Torch:',torch.__version__,'| GPU:',
          torch.cuda.get_device_name(0) if GPU_READY else 'NOT AVAILABLE')
except Exception as ex:
    print('Torch import failed:',type(ex).__name__,str(ex))
print('GPU_READY:',GPU_READY)

missing_packages=[mod for mod in ('yaml','pandas','numpy','scipy','tqdm')
                  if importlib.util.find_spec(mod) is None]
BASIC_DEPS_READY=(len(missing_packages)==0)
print('BASIC_DEPS_READY:',BASIC_DEPS_READY,'| missing:',missing_packages or 'none')
if not BASIC_DEPS_READY:
    print('Colab fix: %pip install pyyaml pandas numpy scipy tqdm')

CONFIG_READY=False
CONFIG_ERROR=''
if REPO_READY:
    try:
        cfg=yaml.safe_load((SOURCE_REPO/'encoder/config/modelconf/lightgcn_plus.yml').read_text(encoding='utf-8'))
        modelcfg=cfg['model']; acfg=modelcfg.get('amazon',{})
        required_checks={
            'embedding_size 32':modelcfg.get('embedding_size')==32,
            'layer_num 3':acfg.get('layer_num',modelcfg.get('layer_num'))==3,
            'kd_weight 0.01':abs(float(acfg.get('kd_weight',modelcfg.get('kd_weight',-1)))-0.01)<1e-12,
            'temperature 0.2':abs(float(acfg.get('kd_temperature',modelcfg.get('kd_temperature',-1)))-0.2)<1e-12,
            'epoch 3000':cfg['train'].get('epoch')==3000,
            'batch_size 4096':cfg['train'].get('batch_size')==4096,
            'patience 5':cfg['train'].get('patience')==5,
            'save_model':cfg['train'].get('save_model') is True,
            'optimizer lr 0.001':abs(float(cfg['optimizer'].get('lr',-1))-0.001)<1e-12,
            'full alignment':str(modelcfg.get('align_mode','full')).lower()=='full',
        }
        failed=[k for k,v in required_checks.items() if not v]
        CONFIG_READY=(not failed)
        if failed: CONFIG_ERROR='Mismatch: '+', '.join(failed)
    except Exception as ex:
        CONFIG_ERROR=f'YAML parsing failed: {ex}'
print('CONFIG_READY:',CONFIG_READY,'|',CONFIG_ERROR or 'OK')

# (5) Combined – mỗi biến phản ánh đúng điều kiện thực
PREFLIGHT_OK=all([REPO_READY,DATA_READY,DATA_LINK_READY,GPU_READY,BASIC_DEPS_READY,CONFIG_READY])
print('\n===== E3C PREFLIGHT =====')
for key in ('REPO_READY','DATA_READY','DATA_LINK_READY','GPU_READY','BASIC_DEPS_READY','CONFIG_READY','PREFLIGHT_OK'):
    print(f'{key}: {globals()[key]}')
if not PREFLIGHT_OK:
    print('\nBLOCKED: Sửa điều kiện FALSE rồi chạy lại cell này; KHÔNG cần chạy training.')
else:
    print('PASS: Có thể thực hiện smoke test/full sweep khi DRY_RUN=False.')


## C. Huấn luyện trong workspace cách ly, không can thiệp E1/E3

- Mỗi run tạo workspace riêng tại `/content/e3c_workspace_<seed>_<weight>`; YAML sửa trong workspace, **không sửa repository gốc**.
- Chạy launcher RLMRec chính thức `python encoder/train_encoder.py --model lightgcn_plus --dataset amazon --cuda 0 --seed ...`.
- Log và checkpoint được sao lưu về Drive, ghi trạng thái `trained_pending_export` nếu thấy checkpoint.
- Không tự train lại run `trained_pending_export`, `passed` hoặc `interrupted_review`; run thất bại cần xem log trước khi đặt `RETRY_FAILED=True`.


In [ ]:
from datetime import datetime, timezone

def write_json(path,obj):
    path.write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str),encoding='utf-8')

def run_single(weight, seed):
    assert weight in WEIGHTS and seed in SEEDS and weight!=DEFAULT_WEIGHT
    if not PREFLIGHT_OK:
        raise RuntimeError('PREFLIGHT chưa PASS. Xem cell 8, không train.')
    dest=run_dir(weight,seed)
    state=get_run_status(weight,seed)
    if state not in ('pending','failed'):
        print(f'SKIP {weight}/{seed} – existing status: {state}')
        return
    if state=='failed' and not RETRY_FAILED:
        print(f'SKIP {weight}/{seed}: failed, cần đọc log rồi mới RETRY_FAILED=True')
        return
    if has_saved_checkpoint(dest):
        raise RuntimeError(f'Đã có checkpoint ở {dest}, không tự ghi đè')

    dest.mkdir(parents=True,exist_ok=True)
    work=Path('/content')/f'e3c_workspace_{weight_slug(weight).replace(".","p")}_{seed}'
    if work.exists():
        raise RuntimeError(f'Workspace còn tồn tại từ run trước: {work}. Kiểm tra trước khi xóa.')

    def exclude(src,names):
        return {n for n in names if n in {'.git','data','checkpoints','checkpoint','logs','log','__pycache__','.ipynb_checkpoints','runs'}}
    shutil.copytree(SOURCE_REPO,work,ignore=exclude,symlinks=True)
    (work/'data').symlink_to(SOURCE_REPO/'data',target_is_directory=True)

    cp=work/'encoder/config/modelconf/lightgcn_plus.yml'
    local_cfg=yaml.safe_load(cp.read_text(encoding='utf-8'))
    assert abs(float(local_cfg['model']['amazon']['kd_weight'])-DEFAULT_WEIGHT)<1e-12
    local_cfg['model']['kd_weight']=float(weight)
    local_cfg['model']['amazon']['kd_weight']=float(weight)
    if 'align_mode' in local_cfg['model']:
        local_cfg['model']['align_mode']='full'
    if 'align_mode' in local_cfg:
        local_cfg['align_mode']='full'
    local_cfg['train']['seed']=int(seed)
    cp.write_text(yaml.safe_dump(local_cfg,sort_keys=False),encoding='utf-8')
    assert yaml.safe_load(cp.read_text())['model']['amazon']['kd_weight']==weight
    (dest/'config_snapshot.yml').write_text(cp.read_text(),encoding='utf-8')

    source_file=SOURCE_REPO/'encoder/models/general_cf/lightgcn_plus.py'
    sha=hashlib.sha256(source_file.read_bytes()).hexdigest() if source_file.is_file() else 'missing'
    meta={'model':MODEL,'dataset':DATASET,'weight':weight,'seed':seed,
          'source_commit':EXPECTED_COMMIT,'model_source_sha256':sha,
          'python_version':sys.version.split()[0],'torch_version':torch.__version__,
          'status':'running','start_utc':datetime.now(timezone.utc).isoformat()}
    write_json(dest/'train_status.json',meta)

    # CLI flag đã đối chiếu với HKUDS/RLMRec/configurator.py
    cmd=[sys.executable,'encoder/train_encoder.py','--model',MODEL,
         '--dataset',DATASET,'--device','cuda','--cuda','0','--seed',str(seed)]
    print(f'RUN lambda={weight} seed={seed}:', ' '.join(cmd),flush=True)
    try:
        with (dest/'training_console.log').open('a',encoding='utf-8') as f:
            f.write('\n========== RUN '+str(datetime.now(timezone.utc))+' ==========\n')
            proc=subprocess.Popen(cmd,cwd=work,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                                  bufsize=1,text=True,errors='replace')
            for line in proc.stdout:
                f.write(line)
                f.flush()
                if any(x in line for x in ('Epoch','Recall','NDCG','Error','Traceback','Best')):
                    print(line.rstrip(),flush=True)
            rc=proc.wait()
        meta['exit_code']=rc
        if rc!=0:
            raise RuntimeError(f'Training exit code={rc}; xem {dest}/training_console.log')

        artifact_paths=[]
        for folder in ('checkpoint','checkpoints','log','logs','save','saved'):
            source=work/folder
            if source.is_dir():
                target=dest/folder
                if target.exists():
                    raise RuntimeError(f'Existing output folder not overwritten: {target}')
                shutil.copytree(source,target)
                artifact_paths.append(str(target))
        checkpoint_files=list(dest.rglob('*.pth'))+list(dest.rglob('*.pt'))
        meta['checkpoint_files']=[str(x) for x in checkpoint_files]
        meta['artifact_paths']=artifact_paths
        meta['status']='trained_pending_export' if checkpoint_files else 'training_finished_check_checkpoint'
        print('RESULT:',meta['status'],'| checkpoints:',len(checkpoint_files))
    except Exception as exc:
        meta['status']='failed';meta['error']=repr(exc)
        print('FAILED:',repr(exc),'\nLOG:',dest/'training_console.log')
        raise
    finally:
        meta['finish_utc']=datetime.now(timezone.utc).isoformat()
        write_json(dest/'train_status.json',meta)
        if work.exists():
            shutil.rmtree(work)


## D. Chạy Smoke Test / Full Sweep

- **Smoke test**: trong cell 3 đặt `DRY_RUN=False`, `SMOKE_TEST_ONLY=True`; chạy lại từ cell 3 đến hết. Chọn run mới đầu tiên, thường là `λ=0, seed=2023`.
- **Full sweep**: sau khi smoke test thành công, đặt `SMOKE_TEST_ONLY=False` rồi chạy lại từ cell 3. Các run `trained_pending_export` được bỏ qua.
- **Không chạy** khi preflight chưa PASS. Các run có lỗi/đứt giữa chừng phải review trước khi thử lại.


In [ ]:
# Làm mới manifest ngay tại thời điểm chạy để không train lại kết quả sẵn có.
manifest_df=build_manifest()
if DRY_RUN:
    print('DRY RUN: chỉ tạo protocol/manifest. Chưa có training.')
elif not PREFLIGHT_OK:
    bad=[x for x in ('REPO_READY','DATA_READY','DATA_LINK_READY','GPU_READY','BASIC_DEPS_READY','CONFIG_READY') if not globals()[x]]
    raise RuntimeError(f'PREFLIGHT BLOCKED: {bad}. Xem thông báo Cell 8 rồi chạy lại Cell 8; chưa train.')
else:
    candidates=[(float(r.weight),int(r.seed)) for r in manifest_df.itertuples()
                if r.status=='pending' or (r.status=='failed' and RETRY_FAILED)]
    candidates=[(w,s) for w,s in candidates if w!=DEFAULT_WEIGHT]
    if SMOKE_TEST_ONLY:
        candidates=candidates[:1]
    print('Planned execution:',candidates)
    for weight,seed in candidates:
        run_single(weight,seed)


## E. Tổng hợp trạng thái và bước kiểm tra sau train

**Chưa được gắn nhãn PASS cuối cùng** nếu chưa dùng exporter Top-K và analyzer E2/E3B, cùng masking train-only. Khi tạo `topk_test.npz`, cần kiểm tra 9.860 users, K=20, không chứa train items, không duplicate, đối chiếu Recall/NDCG với trainer, sau đó mới tính ARP/APLT/TailRecall và dựng accuracy–bias curves.

In [ ]:
rows=[]
for w in WEIGHTS:
    for seed in SEEDS:
        dest=run_dir(w,seed)
        status=get_run_status(w,seed)
        if w==DEFAULT_WEIGHT:
            rows.append(dict(weight=w,seed=seed,status=status,checkpoint='E1 existing',topk=str(ref_topks[seed])))
            continue
        cp_files=(list(dest.rglob('*.pth'))+list(dest.rglob('*.pt'))) if dest.is_dir() else []
        topk=dest/'topk/topk_test.npz'
        rows.append(dict(weight=w,seed=seed,status=status,
             checkpoint=str(cp_files[0]) if cp_files else '',
             topk=str(topk) if topk.is_file() else ''))
status_df=pd.DataFrame(rows)
status_df.to_csv(OUT/'E3C_run_status.csv',index=False)
print(status_df.to_string(index=False))
print('\nNOTE: trained_pending_export != final PASS; cần xuất Top-K và phân tích popularity bias.')


## Kết quả lưu trên Google Drive

```text
Data_SNA_Group14/experiments/E3C/
├── E3C_protocol.json
├── E3C_run_manifest.csv
├── E3C_run_status.csv
└── runs/full/
    ├── weight_0/seed_2023/
    ├── weight_0.005/seed_2023/
    ├── weight_0.02/seed_2023/
    └── weight_0.05/seed_2023/
```

Từng run chứa `config_snapshot.yml`, `train_status.json`, `training_console.log`, và checkpoint nếu trainer có lưu. Đối với **3 seeds λ=0.01**, manifest tham chiếu E1; notebook không train lại.

**Preflight cần PASS đầy đủ** trước khi train. Notebook chưa có pipeline Top-K/Popularity Bias; dùng lại các công cụ E1/E2/E3B với train-only masking để hoàn tất RQ2. Nếu training báo missing dependencies/torch errors, xem `training_console.log` và đối chiếu môi trường E1/E3A, **không tự đổi thuật toán hoặc tự hạ phiên bản package**.
